[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/milioe/casos-ia-ibero-diplomado/blob/main/modulo_4/12_Capacidades_LLM_OpenAI.ipynb)

# Capacidades de un LLM (versión OpenAI)

Este notebook es la versión con **OpenAI** del notebook 12. Recorremos, en un solo lugar, **todo lo que un LLM moderno sabe hacer**: no solo contestar texto.

Usamos **GPT-5.6 Luna** (`gpt-5.6-luna`), el modelo más barato y rápido de la familia GPT-5. A diferencia de Gemini, **OpenAI cobra por cada llamada**: necesitas una clave con saldo.

Vamos en 8 partes:

1. Llamado simple, roles (`sistema`/`usuario`), tokens y **cuánto costó**
2. Reasoning — qué tanto "piensa" el modelo antes de responder
3. Verbosity — qué tan largo responde
4. Visión — ver una imagen y describirla, o leer texto de ella (OCR)
5. Video — partir un video en imágenes ("frames") y dárselas al modelo
6. Audio — texto a voz, voz a texto, y las dos encadenadas
7. Tools — que el modelo llame funciones tuyas, o busque en internet
8. Structured Outputs — que te regrese JSON con una estructura exacta

## Archivos que necesitas

1. `calorias.jpeg`
2. `supermarket.mp4`
3. `Greece.mp3`

## Setup — instalar librerías

- `openai` — cliente de la API de OpenAI.
- `opencv-python-headless` — leer el video y sacarle frames.
- `pillow` — abrir y mostrar imágenes.
- `pydantic` — definir el esquema de los Structured Outputs.

In [ ]:
%pip install -q openai opencv-python-headless pillow requests pydantic

## Clave API (OpenAI, desde Colab)

1. Entra a [OpenAI Platform → API keys](https://platform.openai.com/api-keys) y crea una clave. Tu cuenta necesita saldo (Billing).
2. En Colab, abre el panel de **Secrets** (ícono de llave 🔑 en la barra lateral izquierda) y guárdala con el nombre `OPENAI_API_KEY`. Actívala para este notebook.

In [ ]:
import base64
import io
import json
from typing import List

import cv2
import matplotlib.pyplot as plt
import requests
from google.colab import userdata
from IPython.display import Audio, display
from openai import OpenAI
from PIL import Image
from pydantic import BaseModel

plt.rcParams.update({"figure.dpi": 100, "axes.grid": False})

API_KEY = userdata.get("OPENAI_API_KEY").strip()

cliente = OpenAI(api_key=API_KEY)

MODELO = "gpt-5.6-luna"
MODELO_TTS = "gpt-4o-mini-tts"
MODELO_STT = "gpt-transcribe"

# Precios en dólares por cada millón de tokens (developers.openai.com/api/docs/pricing).
PRECIO_ENTRADA = 0.20
PRECIO_SALIDA = 1.20


def costo(respuesta):
    """Cuánto costó una llamada a responses.create, en dólares."""
    uso = respuesta.usage
    return (uso.input_tokens * PRECIO_ENTRADA + uso.output_tokens * PRECIO_SALIDA) / 1_000_000


def imagen_a_url(imagen):
    """Convierte una imagen PIL en un data URL, que es como OpenAI recibe imágenes locales."""
    buffer = io.BytesIO()
    imagen.convert("RGB").save(buffer, format="JPEG", quality=85)
    return "data:image/jpeg;base64," + base64.b64encode(buffer.getvalue()).decode()


print(f"Cliente listo. Modelo: {MODELO}")

## 1 — Llamado simple, roles y tokens

Lo mínimo: un texto de entrada, un texto de salida.

In [ ]:
respuesta = cliente.responses.create(
    model=MODELO,
    input="Hola! ¿Cómo estás?",
)

print(respuesta.output_text)

### Prompt de sistema, usuario y asistente

`instructions` fija cómo se comporta el modelo durante toda la conversación (el "prompt de sistema"). Los roles `user`/`assistant` arman el historial — así el modelo "recuerda" lo que ya se dijo.

In [ ]:
respuesta = cliente.responses.create(
    model=MODELO,
    instructions=(
        "Eres un asistente de IA que ayuda a los alumnos de la Ibero a resolver sus dudas. "
        "Siempre preséntate como Iber.ia y pregunta el nombre del alumno."
    ),
    input=[
        {"role": "user", "content": "Hola!"},
        {"role": "assistant", "content": "Hola! Soy Iber.ia, tu asistente virtual. ¿Cómo te llamas?"},
        {"role": "user", "content": "Emilio, ¿tú cómo te llamas?"},
    ],
)

print(respuesta.output_text)

### Cuántos tokens usó y cuánto costó

Cada llamada trae el conteo de tokens en `usage`. OpenAI cobra por separado los tokens de entrada y los de salida, así que con eso sabemos exactamente cuánto costó.

In [ ]:
print(f"Tokens de entrada:  {respuesta.usage.input_tokens}")
print(f"Tokens de salida:   {respuesta.usage.output_tokens}")
print(f"Tokens totales:     {respuesta.usage.total_tokens}")
print(f"Costo:              ${costo(respuesta):.6f} dólares")

## 2 — Reasoning: qué tanto "piensa" antes de responder

Los modelos de razonamiento generan pasos intermedios antes de la respuesta final — útil en matemáticas, lógica o planeación. Con `reasoning={"effort": ...}` controlas qué tanto: `low`, `medium` o `high`. Más esfuerzo = respuestas mejor razonadas, pero más lentas y con más tokens (que también se cobran).

Fíjate en los **tokens de razonamiento** que imprime cada celda: no los ves en la respuesta, pero sí los pagas.

In [ ]:
respuesta = cliente.responses.create(
    model=MODELO,
    input="""
    Vas a comprar 3 cuadernos de $45 y 2 plumas de $18.
    En la Tienda A hacen 15% de descuento sobre el subtotal y luego aplican IVA de 16%.
    En la Tienda B hacen 10% de descuento a cada artículo por separado y luego aplican IVA de 16%.
    ¿En cuál tienda pagas menos? Da el total final de cada tienda y concluye con una verificación breve (1 oración).
    """,
    reasoning={"effort": "low"},
)

print(respuesta.output_text)
print(f"\nTokens de razonamiento: {respuesta.usage.output_tokens_details.reasoning_tokens} | Costo: ${costo(respuesta):.6f}")

In [ ]:
respuesta = cliente.responses.create(
    model=MODELO,
    input="""
    ¿Cuántas pelotas de tenis caben en un salón de clases de 8 m x 6 m x 3 m?
    Indica tres supuestos clave (p. ej., diámetro de la pelota, factor de empaque, espacio ocupado por muebles),
    muestra un cálculo aproximado y reporta un rango razonable (entre la mitad y el doble de tu estimación).
    Termina con la cifra que considerarías más plausible.
    """,
    reasoning={"effort": "medium"},
)

print(respuesta.output_text)
print(f"\nTokens de razonamiento: {respuesta.usage.output_tokens_details.reasoning_tokens} | Costo: ${costo(respuesta):.6f}")

In [ ]:
respuesta = cliente.responses.create(
    model=MODELO,
    input="""
    Programa estas tres clases en una mañana con dos salones (Azul y Verde) y tres franjas (8-9, 9-10, 10-11):
    Álgebra (A), Biología (B), Historia (H).
    Profes: Lara (puede A o H; no disponible 9-10), Kim (solo B; disponible todo), Soto (solo A; disponible 9-10 o 10-11).
    Reglas: B debe impartirse en el salón Verde; Historia no puede ir 9-10; no puede haber dos clases en el mismo
    salón y franja; cada clase se imparte exactamente una vez.
    Entrega una tabla final (franja, salón, clase, profesor) y 2-3 razones clave que muestren por qué tu asignación
    cumple todas las restricciones. Si fuera imposible, dilo y explica por qué en una frase.
    """,
    reasoning={"effort": "high"},
)

print(respuesta.output_text)
print(f"\nTokens de razonamiento: {respuesta.usage.output_tokens_details.reasoning_tokens} | Costo: ${costo(respuesta):.6f}")

## 3 — Verbosity: qué tan largo responde

**Verbosity** controla la cantidad de texto que genera el modelo — respuestas cortas y directas vs. largas y detalladas. En GPT-5 es un parámetro: `text={"verbosity": "low" | "medium" | "high"}`.

Misma pregunta, dos niveles:

In [ ]:
pregunta = "¿Qué es una red neuronal?"

for nivel in ["low", "high"]:
    respuesta = cliente.responses.create(
        model=MODELO,
        input=pregunta,
        text={"verbosity": nivel},
    )
    print(f"--- verbosity={nivel} ({respuesta.usage.output_tokens} tokens de salida) ---")
    print(respuesta.output_text)
    print()

## 4 — Visión

### Una imagen de internet

In [ ]:
HEADERS_WIKIMEDIA = {"User-Agent": "IberoDiplomadoNLP/1.0 (uso educativo, sin fines comerciales)"}

resp_img = requests.get(
    "https://commons.wikimedia.org/wiki/Special:FilePath/The_Parthenon_in_Athens.jpg?width=600",
    headers=HEADERS_WIKIMEDIA,
    timeout=20,
)
resp_img.raise_for_status()
imagen_parthenon = Image.open(io.BytesIO(resp_img.content))

respuesta = cliente.responses.create(
    model=MODELO,
    input=[{
        "role": "user",
        "content": [
            {"type": "input_text", "text": "Dime qué ves en esta imagen"},
            {"type": "input_image", "image_url": imagen_a_url(imagen_parthenon)},
        ],
    }],
)

display(imagen_parthenon)
print(respuesta.output_text)

### Una imagen propia — OCR

Le pedimos que lea la tabla de información nutricional de `calorias.jpeg` y nos la regrese en formato markdown.

In [ ]:
imagen_nutricion = Image.open("calorias.jpeg")

respuesta = cliente.responses.create(
    model=MODELO,
    input=[{
        "role": "user",
        "content": [
            {"type": "input_text", "text": "Extrae la tabla de información nutricional de esta imagen y devuélvela en formato markdown."},
            {"type": "input_image", "image_url": imagen_a_url(imagen_nutricion)},
        ],
    }],
)

display(imagen_nutricion)
print(respuesta.output_text)

## 5 — Video: partirlo en frames

OpenAI no recibe video directo en este modelo, así que hacemos **la técnica a mano**: sacar una foto del video cada N segundos y mandar esas fotos como si fueran una serie de imágenes.

Ojo con el costo: **cada foto se cobra como tokens de entrada**. Más fotos = más caro.

In [ ]:
def video_a_frames(ruta_video, cada_n_segundos=3):
    """Saca un frame del video cada N segundos y lo regresa como imagen PIL."""
    video = cv2.VideoCapture(ruta_video)
    fps = video.get(cv2.CAP_PROP_FPS)
    intervalo = int(fps * cada_n_segundos)

    frames = []
    contador = 0
    while True:
        ok, frame = video.read()
        if not ok:
            break
        if contador % intervalo == 0:
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(Image.fromarray(frame_rgb))
        contador += 1
    video.release()
    return frames


frames = video_a_frames("supermarket.mp4", cada_n_segundos=3)
print(f"Frames extraídos: {len(frames)}")

fig, axes = plt.subplots(2, 5, figsize=(16, 6.5))
for ax, frame in zip(axes.flat, frames):
    ax.imshow(frame)
    ax.axis("off")
for ax in axes.flat[len(frames):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
prompt_video = """
A continuación se te dan fotos tomadas de un video de un supermercado. Extrae los productos con sus precios
en este formato:

Producto:
Precio:
----

Reglas:
- No inventes información.
- No repitas productos si aparecen en más de una foto.
- Si el precio no es visible en una foto, revisa las fotos cercanas; si de plano no aparece, escribe "No visible".
"""

contenido = [{"type": "input_text", "text": prompt_video}]
for frame in frames:
    contenido.append({"type": "input_image", "image_url": imagen_a_url(frame)})

respuesta = cliente.responses.create(
    model=MODELO,
    input=[{"role": "user", "content": contenido}],
)

print(respuesta.output_text)
print(f"\nTokens de entrada: {respuesta.usage.input_tokens} | Costo: ${costo(respuesta):.6f}")

## 6 — Audio

### Texto a voz (TTS)

OpenAI tiene modelos especiales para audio. Aquí usamos `gpt-4o-mini-tts`, que regresa directo un archivo `.mp3`.

In [ ]:
with cliente.audio.speech.with_streaming_response.create(
    model=MODELO_TTS,
    voice="coral",
    input="Hola a todos los participantes del diplomado de IA de Ibero!",
) as respuesta_audio:
    respuesta_audio.stream_to_file("salida_tts.mp3")

print("Guardado: salida_tts.mp3")
Audio("salida_tts.mp3")

### Voz a texto (STT)

Le damos `Greece.mp3` a `gpt-transcribe` para que lo transcriba, y luego le pedimos a GPT que traduzca el texto al español.

In [ ]:
with open("Greece.mp3", "rb") as f:
    transcripcion = cliente.audio.transcriptions.create(model=MODELO_STT, file=f)

print("Transcripción:", transcripcion.text)

traduccion = cliente.responses.create(
    model=MODELO,
    input=f"Traduce al español este texto:\n\n{transcripcion.text}",
)
print("\nTraducción:", traduccion.output_text)

### "Habla con GPT" — las dos encadenadas

Lo hacemos en **dos pasos**: texto → texto, y luego texto → audio. OpenAI también tiene modelos que reciben y responden audio en una sola llamada (`gpt-realtime`), pero son bastante más caros.

In [ ]:
pregunta = "Hola! ¿Cómo estás? Me llamo Emilio."

respuesta_texto = cliente.responses.create(model=MODELO, input=pregunta)
print("Respuesta en texto:", respuesta_texto.output_text)

with cliente.audio.speech.with_streaming_response.create(
    model=MODELO_TTS,
    voice="coral",
    input=respuesta_texto.output_text,
) as respuesta_audio:
    respuesta_audio.stream_to_file("respuesta_hablada.mp3")

Audio("respuesta_hablada.mp3")

## 7 — Tools

### Function calling

Le damos al modelo una función de Python (`obtener_clima`) descrita como "herramienta". El modelo decide **si** y **cómo** llamarla — nosotros la ejecutamos y le devolvemos el resultado.

In [ ]:
def obtener_clima(latitud, longitud):
    resp = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={"latitude": latitud, "longitude": longitud, "current": "temperature_2m"},
        timeout=20,
    )
    return resp.json()["current"]["temperature_2m"]


herramienta_clima = {
    "type": "function",
    "name": "obtener_clima",
    "description": "Obtiene la temperatura actual en grados Celsius para unas coordenadas.",
    "parameters": {
        "type": "object",
        "properties": {
            "latitud": {"type": "number"},
            "longitud": {"type": "number"},
        },
        "required": ["latitud", "longitud"],
        "additionalProperties": False,
    },
    "strict": True,
}

pregunta_clima = "¿Cuál es el clima en la CDMX, Tlalpan?"

respuesta = cliente.responses.create(
    model=MODELO,
    input=pregunta_clima,
    tools=[herramienta_clima],
)

llamada = next(item for item in respuesta.output if item.type == "function_call")
argumentos = json.loads(llamada.arguments)
print(f"El modelo quiere llamar a: {llamada.name}({argumentos})")

In [ ]:
resultado = obtener_clima(**argumentos)
print(f"Temperatura real: {resultado}°C")

respuesta_final = cliente.responses.create(
    model=MODELO,
    previous_response_id=respuesta.id,
    input=[{
        "type": "function_call_output",
        "call_id": llamada.call_id,
        "output": json.dumps({"temperatura_celsius": resultado}),
    }],
    tools=[herramienta_clima],
)

print(respuesta_final.output_text)

### Tools ya armadas: buscar en internet

No todas las herramientas las escribes tú — OpenAI trae **búsqueda web** integrada, para preguntas que necesitan información de ahorita. Además de los tokens, **cada búsqueda cuesta 1 centavo de dólar**.

In [ ]:
respuesta = cliente.responses.create(
    model=MODELO,
    input="Háblame sobre los mayores eventos económicos de este mes en México",
    tools=[{"type": "web_search"}],
)

print(respuesta.output_text)

## 8 — Structured Outputs

En vez de texto libre, le pedimos al modelo que regrese **JSON con una forma exacta** — útil para meter la respuesta directo a una base de datos o a otro sistema. Definimos esa forma con una clase de `pydantic`.

In [ ]:
class InformacionPersonal(BaseModel):
    nombre_completo: str
    correo: str
    telefono: str
    ubicacion: str


class Experiencia(BaseModel):
    puesto: str
    empresa: str
    fecha_inicio: str
    fecha_fin: str
    responsabilidades: List[str]


class Educacion(BaseModel):
    grado: str
    institucion: str
    anio_fin: int


class CurriculumInfo(BaseModel):
    informacion_personal: InformacionPersonal
    anios_experiencia: int
    experiencia_laboral: List[Experiencia]
    educacion: List[Educacion]
    habilidades_tecnicas: List[str]

In [ ]:
curriculum_texto = """
CURRÍCULUM VITAE

Nombre: Carlos Alberto Martínez López
Correo: carlos.martinez@datamail.com | Teléfono: +52 55 1234 5678
Ubicación: CDMX, México

PERFIL PROFESIONAL
Ingeniero de Datos con 5 años de experiencia en arquitecturas de datos escalables, Big Data y soluciones cloud.

EXPERIENCIA LABORAL

Senior Data Engineer | TechCorp México | Enero 2022 - Presente
- Diseño de pipelines procesando 10TB diarios con Apache Spark y Airflow
- Implementación de data lake en AWS (S3, Glue, Athena)
- Migración a AWS reduciendo costos 40%

Data Engineer | DataSolutions | Marzo 2020 - Diciembre 2021
- Desarrollo de ETL pipelines en Python
- Optimización de SQL mejorando tiempos 60%
- Dashboards en Tableau para stakeholders

EDUCACIÓN
- Maestría en Ciencia de Datos | UNAM | 2021-2023
- Ingeniería en Sistemas | ITESM | 2015-2019

HABILIDADES TÉCNICAS
Python, SQL, Scala, Spark, Kafka, Airflow, AWS, Docker, Kubernetes, Tableau
"""

respuesta = cliente.responses.parse(
    model=MODELO,
    input=f"Extrae la información de este currículum:\n\n{curriculum_texto}",
    text_format=CurriculumInfo,
)

curriculum = respuesta.output_parsed
print(curriculum.model_dump_json(indent=2))

## Cierre

Un LLM moderno no es solo "un chat". Con el mismo cliente sirve para razonar, controlar qué tan largo responde, leer imágenes y videos, hablar y escuchar, llamar funciones tuyas, buscar en internet y devolver datos con una forma exacta.

**Lo que no cubrimos aquí:**

- Generación de imágenes: OpenAI sí la tiene (`gpt-image`), pero es de lo más caro de la API.
- Audio-a-audio en una sola llamada (`gpt-realtime`): también bastante más caro.

La diferencia práctica con la versión de Gemini es el **costo**: aquí cada llamada se paga, así que conviene revisar `usage` y elegir el modelo más barato que resuelva la tarea.